# Tahap 8 · Augmentasi konteks TabPFN-3.5

Tambah **dataset Kaggle tahap 7** yang berisi `label_01/`, `label_10/`, `label_30/`.
Aktifkan **GPU T4 x2** dan **Internet**. Kode ada di notebook. Bobot 3.5 diunduh sekali.

Tujuan: ukur nilai SSL dari unlabeled, lalu tambahan manfaat masing-masing augmentasi.
Smoke aktif saat pertama dibuka. Setelah selesai, set `RUN_FULL = True`.

In [ ]:
%%bash
set -euo pipefail
nvidia-smi -L
python -m pip install -q 'tabpfn==10.2.0'

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

WORK = Path("/kaggle/working")
OUT = WORK / "tahap8"
os.environ["TABPFN_NO_BROWSER"] = "1"
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
# Isi jika ada lebih dari satu dataset split di /kaggle/input.
os.environ["DATA_DIR"] = ""
# Opsional. Hub model publik dapat diunduh langsung tanpa token Hugging Face.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("TABPFN_TOKEN")
except Exception:
    token = None
if token:
    os.environ["TABPFN_TOKEN"] = token
del token

RUN_SMOKE = True
RUN_FULL = False
RUN_ABLATIONS = False  # tambah grid kekuatan konteks setelah run utama
FULL_ARGS = []  # contoh pilot skala penuh: ["--datasets", "superconductivity", "--percents", "30", "--seeds", "42"]
RESUME_ZIP = ""  # opsional: path arsip tahap8 yang ditambahkan ke Kaggle Input
EXPORT_CACHE = False  # True untuk membawa embedding cache sesi berikutnya

if RESUME_ZIP:
    with ZipFile(RESUME_ZIP) as archive:
        for entry in archive.infolist():
            target = (OUT / entry.filename).resolve()
            if entry.is_dir() or not target.is_relative_to(OUT.resolve()) or target.exists():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            partial = target.with_suffix(".restore_tmp")
            with archive.open(entry) as source, partial.open("wb") as sink:
                shutil.copyfileobj(source, sink)
            partial.replace(target)

def run(file, *args):
    subprocess.run([sys.executable, "-u", str(WORK / file), *map(str, args)], check=True)

In [ ]:
%%writefile /kaggle/working/stage8_common.py
"""Helper yang disalin oleh buat_notebook.py dari tahap 5."""

import math

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import (accuracy_score, cohen_kappa_score, f1_score, mean_absolute_error,
                             mean_squared_error, r2_score)


def load(root, name):
    with np.load(root / f'{name}.npz', allow_pickle=False) as split:
        return split['X'].astype(np.float32), (split['y'] if 'y' in split else None)


def scores(task, y, pred, levels=None):
    if task == 'classification':
        return {'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0)),
                'accuracy': float(accuracy_score(y, pred))}
    result = {'mse': float(mean_squared_error(y, pred)), 'mae': float(mean_absolute_error(y, pred)),
              'r2': float(r2_score(y, pred)), 'spearman': float(spearmanr(y, pred).statistic)}
    if task == 'ordinal_regression':
        rounded = levels[np.abs(pred.reshape(-1, 1) - levels).argmin(axis=1)]
        result.update(qwk=float(cohen_kappa_score(y, rounded, labels=levels, weights='quadratic')),
                      ordinal_mae=float(mean_absolute_error(y, rounded)),
                      accuracy=float(accuracy_score(y, rounded)))
    return result


def same_rank_donors(pseudo_labels: torch.Tensor, confidence: torch.Tensor, cutoff: float):
    n = len(pseudo_labels)
    eligible = (pseudo_labels[:, None].eq(pseudo_labels[None, :])
                & confidence[None, :].ge(cutoff)
                & ~torch.eye(n, dtype=torch.bool, device=pseudo_labels.device))
    has_donor = eligible.any(dim=1)
    priorities = torch.rand((n, n), device=pseudo_labels.device)
    donors = priorities.masked_fill(~eligible, -1.0).argmax(dim=1)
    return torch.where(has_donor, donors, torch.arange(n, device=pseudo_labels.device)), has_donor


def rotate(h: torch.Tensor, u: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """h' = ‖h‖ (c·v + √(1−c²)·û), v = h/‖h‖, û = bagian u yang tegak lurus v (dinormalkan). Norma tetap."""
    norm = h.norm(dim=-1, keepdim=True)
    v = h / norm.clamp_min(1e-12)
    u = u - (u * v).sum(-1, keepdim=True) * v
    u = u / u.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    rotated = norm * (c * v + (1 - c ** 2).clamp_min(0).sqrt() * u)
    return torch.where(norm > 1e-6, rotated, h)


def angular(h: torch.Tensor, low: float, high: float) -> torch.Tensor:
    """Putar h ke arah acak sehingga cos(h, h') = c ~ U(low, high) tepat, norma tetap (ide pembimbing)."""
    u = torch.randn_like(h)  # urutan RNG sama dengan versi lama: hasil run angular lama tetap bisa diulang
    return rotate(h, u, torch.empty_like(h[..., :1]).uniform_(low, high))


def head_view(weight: torch.Tensor, rank: bool) -> torch.Tensor:
    """Bobot head (k, d, out) yang benar-benar dilihat loss. RankUp: output [regresi, ARC0, ARC1] dan
    peluang i > j hanya bergantung pada r_i − r_j, dengan r = h·a, a = W[..., 2] − W[..., 1]."""
    weight = weight.detach()
    return (weight[..., 2] - weight[..., 1])[..., None] if rank else weight


def head_direction(h: torch.Tensor, weight: torch.Tensor, rank: bool, adversarial: bool) -> torch.Tensor:
    """Arah di ruang h yang paling mengubah output head linear (closed form, tanpa backward tambahan).

    rank: skor r = h·a didorong ke median batch, sehingga urutan antar sampel paling sulit dibedakan.
    klasifikasi: dari bobot kelas teratas ke kelas kedua (batas keputusan terdekat).
    adversarial=False: tanda arah diacak (kontrol 'headrot': sadar-head tapi tidak adversarial).
    """
    w = head_view(weight, rank)                                       # (k, d, o)
    if rank:
        a = w[..., 0]                                                 # (k, d)
        r = (h * a).sum(-1)                                           # (B, k)
        sign = -torch.sign(r - r.median(dim=0).values)
        direction = a.expand_as(h)
    else:
        top = torch.einsum('bkd,kdo->bko', h, w).topk(2, dim=-1).indices
        wt, member = w.transpose(1, 2), torch.arange(w.shape[0], device=h.device)  # (k, o, d)
        direction = wt[member, top[..., 1]] - wt[member, top[..., 0]]  # (B, k, d)
        sign = torch.ones(h.shape[:2], device=h.device)
    if not adversarial:
        sign = torch.randint(0, 2, h.shape[:2], device=h.device) * 2.0 - 1
    sign = torch.where(sign == 0, torch.ones_like(sign), sign)        # sampel tepat di median
    return sign[..., None] * direction


def perturb(op: str, h: torch.Tensor, strength, pseudo_labels=None, confidence=None,
            donor_confidence: float = 0.8, head=None, rank: bool = True) -> torch.Tensor:
    """h: (batch, submodel, dim). strength: angka; (low, high) cosine untuk 'angular';
    (low, high) angular distance d ∈ [0, 1] (θ = π·d) untuk 'advrot'/'headrot', yang juga butuh head (k, d, out)."""
    chosen = torch.rand_like(h) < strength if op in ('zero_out', 'scarf', 'tabcutmix') else None
    if op == 'gaussian':  # skala noise ikut sebaran tiap unit di batch, jadi strength tanpa satuan
        return h + strength * h.std(dim=0, keepdim=True, unbiased=False) * torch.randn_like(h)
    if op == 'zero_out':
        return h.masked_fill(chosen, 0.0)
    if op == 'scarf':  # unit terpilih diambil dari baris acak lain di batch
        return torch.where(chosen, h.gather(0, torch.randint(len(h), h.shape, device=h.device)), h)
    if op == 'tabcutmix':  # unit terpilih diambil dari baris dengan pseudo-label sama
        donors, has_donor = same_rank_donors(pseudo_labels, confidence, donor_confidence)
        return torch.where(chosen & has_donor[:, None, None], h[donors], h)
    if op == 'dropout':  # UniMatch / CCT
        return F.dropout(h, p=strength, training=True)
    if op == 'fnoise':  # CCT F-Noise: z + z·U(−s, s)
        return h + h * (2 * torch.rand_like(h) - 1) * strength
    if op == 'angular':
        return angular(h, *strength)
    if op in ('advrot', 'headrot'):
        d = torch.empty_like(h[..., :1]).uniform_(*strength)
        return rotate(h, head_direction(h, head, rank, op == 'advrot'), torch.cos(math.pi * d))
    raise ValueError(f'operator latent tidak dikenal: {op!r}')

In [ ]:
%%writefile /kaggle/working/tabpfn_konteks.py
"""Latih head SSL dengan in-context subsetting di atas TabPFN yang dibekukan."""

from __future__ import annotations

import argparse
import fcntl
import hashlib
import json
import os
import subprocess
import sys
import time
from copy import deepcopy
from importlib.metadata import version
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn

from stage8_common import load, perturb, rotate, scores

HERE = Path(__file__).resolve().parent
DATASETS = ("nhanes", "pendigits", "superconductivity")
PERCENTS = (1, 10, 30)
SEEDS = (42, 123, 456)
METRIC = {"nhanes": ("qwk_calibrated", 1), "pendigits": ("macro_f1", 1), "superconductivity": ("mse", -1)}
SUPERVISED = ("tabpfn", "head_sup", "ics_sup")
SSL_IDENTITY = ("head_distill", "head_match")
LATENT_DISTILL = {"head_gaussian": ("gaussian", 0.5), "head_dropout": ("dropout", 0.5),
                  "head_fnoise": ("fnoise", 0.3), "head_tabcutmix": ("tabcutmix", 0.5),
                  "head_angular": ("angular", (0.3, 0.5)), "head_scarf": ("scarf", 0.5)}
SSL_AUGMENTED = (*LATENT_DISTILL, "head_match_gaussian", "ics_distill_70", "ics_match_70", "ics_both_70")
CORE = (*SUPERVISED, *SSL_IDENTITY, *SSL_AUGMENTED)
ABLATIONS = ("ics_distill_50", "ics_distill_90", "ics_distill_ta")
VARIANTS = (*CORE, *ABLATIONS)
MATCH_VARIANTS = ("head_match", "head_match_gaussian", "ics_match_70")
EMA_DECAY, CONFIDENCE = 0.99, 0.95
WEIGHT_REPO = "Prior-Labs/tabpfn_3_5"
WEIGHT_NAME = "tabpfn-v3.5-20260909.safetensors"
TABPFN_VERSION = "10.2.0"
SPLITS = ("train_labeled", "train_unlabeled", "validation", "test")
SCHEMA = 2


def eligible(names, percent):
    return [name for name in names if percent < 30 or name not in ("ics_distill_50", "ics_distill_90")]


def experiment_names(choice):
    if choice == "core":
        return CORE
    if choice == "all":
        return VARIANTS
    return parse_list(choice, VARIANTS)


def find_data(explicit):
    if explicit:
        candidates = [Path(explicit).expanduser()]
    else:
        candidates = [HERE.parent / "5-augmentasi-latent/data"]
        candidates += [p.parent for p in sorted(Path("/kaggle/input").rglob("label_01"))]
    for path in candidates:
        if all(part.is_file() for part in split_files(path)):
            return path.resolve()
    raise SystemExit("Split lengkap tidak ditemukan. Tambahkan dataset Kaggle tahap 7 atau set DATA_DIR.")


def read_data(data_root, dataset, percent):
    root = data_root / f"label_{percent:02d}" / dataset
    manifest = json.loads((root / "manifest.json").read_text())
    task = manifest["task_type"]
    class_ids = {str(c): i for i, c in enumerate(manifest.get("classes", []))}
    levels = np.asarray(manifest["target_values"]) if task == "ordinal_regression" else None
    splits = {}
    for name in SPLITS:
        x, y = load(root, name)
        if name == "train_unlabeled":
            splits[name] = (x, None)
        elif task == "classification":
            splits[name] = (x, np.array([class_ids[c] for c in y.astype(str)], dtype=np.int64))
        else:
            splits[name] = (x, np.asarray(y, dtype=np.float32).reshape(-1))
    return task, splits, levels


def choose_unlabeled(n, limit, seed):
    if not limit or n <= limit:
        return np.arange(n)
    return np.sort(np.random.default_rng(seed).choice(n, size=limit, replace=False))


def context_strata(y, task):
    if task != "regression":
        return y
    edges = np.unique(np.quantile(y, np.linspace(0.1, 0.9, 9)))
    return np.searchsorted(edges, y, side="right")


def stratified_subset(y, task, rho, rng):
    """Proportional allocation with at least one row per stratum present in the pool."""
    _, inverse, counts = np.unique(context_strata(y, task), return_inverse=True, return_counts=True)
    total = max(len(counts), round(len(y) * rho))
    target = counts * (total / len(y))
    quota = np.maximum(1, np.floor(target).astype(int))
    while quota.sum() > total:
        excess = np.where(quota > 1, quota - target, -np.inf)
        quota[excess.argmax()] -= 1
    while quota.sum() < total:
        deficit = np.where(quota < counts, target - quota, -np.inf)
        quota[deficit.argmax()] += 1
    rows = [rng.choice(np.flatnonzero(inverse == k), size=n, replace=False) for k, n in enumerate(quota)]
    # Keep relative input order so rho=1 is the original context.
    return np.sort(np.concatenate(rows))


def folds(x, y, task, seed):
    # Same splitter as TabPFNEmbedding.fit_transform in stage 7.
    splitter = StratifiedKFold if task == "classification" else KFold
    return list(splitter(n_splits=5, shuffle=True, random_state=seed).split(x, y))


def weights(out, override):
    if override:
        path = Path(override).expanduser().resolve(strict=True)
        if path.name != WEIGHT_NAME:
            raise SystemExit(f"File bobot harus bernama {WEIGHT_NAME}")
        return path
    from huggingface_hub import hf_hub_download

    return Path(hf_hub_download(WEIGHT_REPO, WEIGHT_NAME, local_dir=out / "weights"))


def make_model(task, seed, model_path, device):
    from tabpfn import TabPFNClassifier, TabPFNRegressor

    cls = TabPFNClassifier if task == "classification" else TabPFNRegressor
    return cls(model_path=model_path, device=device, random_state=seed)


def chunked(fn, x, size):
    return np.concatenate([fn(x[s:s + size]) for s in range(0, len(x), size)], axis=0)


def embedding(model, x, chunk):
    def run(rows):
        members = np.asarray(model.get_embeddings(rows, data_source="test"))
        if members.ndim != 3 or members.shape[1] != len(rows):
            raise ValueError(f"Shape embedding TabPFN tidak dikenal: {members.shape}")
        return members.mean(axis=0, dtype=np.float32)

    return chunked(run, x, chunk)


def teacher(model, x, task, chunk):
    infer = model.predict_proba if task == "classification" else model.predict
    return chunked(infer, x, chunk).astype(np.float32)


def build_base(new_model, task, splits, unlabeled_rows, seed, chunk):
    x, y = splits["train_labeled"]
    model = new_model()
    labeled = []
    for context, query in folds(x, y, task, seed):
        model.fit(x[context], y[context])
        labeled.append((query, embedding(model, x[query], chunk)))
    order = np.argsort(np.concatenate([query for query, _ in labeled]))
    base = {"e_lab": np.concatenate([emb for _, emb in labeled])[order], "unlabeled_rows": unlabeled_rows}
    model.fit(x, y)
    for name, query in (
        ("unl", splits["train_unlabeled"][0][unlabeled_rows]),
        ("val", splits["validation"][0]), ("test", splits["test"][0]),
    ):
        base[f"e_{name}"] = embedding(model, query, chunk)
        base[f"t_{name}"] = teacher(model, query, task, chunk)
    return base


def build_views(new_model, task, splits, unlabeled_rows, seed, chunk, views, setting):
    x, y = splits["train_labeled"]
    model = new_model()
    labeled = setting == "lab70"
    fold_pairs = folds(x, y, task, seed) if labeled else [(np.arange(len(x)), unlabeled_rows)]
    all_embeddings, all_predictions, all_rhos, all_sizes = [], [], [], []
    # Separate deterministic streams for view settings and folds.
    setting_id = {"50": 50, "70": 70, "90": 90, "ta": 100, "lab70": 170}[setting]
    rng = np.random.default_rng(np.random.SeedSequence([seed, setting_id]))
    for _ in range(views):
        rho = float(rng.uniform(0.5, 0.99)) if setting == "ta" else int(setting[-2:]) / 100
        view, targets, sizes = [], [], []
        for context, query in fold_pairs:
            used = context[stratified_subset(y[context], task, rho, rng)]
            model.fit(x[used], y[used])
            rows = x[query] if labeled else splits["train_unlabeled"][0][query]
            view.append(embedding(model, rows, chunk))
            if not labeled:
                targets.append(teacher(model, rows, task, chunk))
            sizes.append(len(used))
        merged = np.concatenate(view)
        if labeled:
            merged = merged[np.argsort(np.concatenate([q for _, q in fold_pairs]))]
        all_embeddings.append(merged)
        if targets:
            all_predictions.append(np.concatenate(targets))
        all_rhos.append(rho)
        all_sizes.append(sizes)
    result = {"embedding": np.stack(all_embeddings), "rho": np.array(all_rhos), "context_sizes": np.array(all_sizes)}
    if all_predictions:
        result["teacher"] = np.stack(all_predictions)
    return result


def atomic_npz(path, arrays, compress=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with tmp.open("wb") as sink:
        save = np.savez_compressed if compress else np.savez
        save(sink, **arrays)
        sink.flush()
        os.fsync(sink.fileno())
    tmp.replace(path)


def cached(path, build):
    if path.exists():
        with np.load(path, allow_pickle=False) as source:
            return dict(source)
    start = time.monotonic()
    result = build()
    result["build_seconds"] = np.asarray(time.monotonic() - start)
    atomic_npz(path, result)
    print(f"{path.parent.name}/{path.stem}: {float(result['build_seconds']):.1f} s", flush=True)
    return result


def calibrate(y_val, pred_val, levels):
    proportions = np.cumsum([(y_val == level).mean() for level in levels])[:-1]
    return np.quantile(pred_val, proportions)


def evaluate(task, levels, y_val, y_test, pred_val, pred_test):
    classification = task == "classification"
    raw_val = pred_val.argmax(axis=1) if classification else pred_val
    raw_test = pred_test.argmax(axis=1) if classification else pred_test
    arrays = {"y_validation": y_val, "y_test": y_test, "pred_validation": raw_val, "pred_test": raw_test}
    if classification:
        arrays.update(proba_validation=pred_val, proba_test=pred_test)
    results = {split: scores(task, arrays[f"y_{split}"], arrays[f"pred_{split}"], levels)
               for split in ("validation", "test")}
    if levels is not None:
        cutoffs = calibrate(y_val, raw_val, levels)
        arrays.update(levels=levels, cutoffs=cutoffs)
        for split in ("validation", "test"):
            calibrated = levels[np.searchsorted(cutoffs, arrays[f"pred_{split}"], side="right")]
            arrays[f"pred_{split}_calibrated"] = calibrated
            results[split]["qwk_calibrated"] = float(cohen_kappa_score(
                arrays[f"y_{split}"], calibrated, labels=levels, weights="quadratic"
            ))
    for values in results.values():
        for key, value in values.items():
            if not np.isfinite(value):
                values[key] = None
    return results, arrays


def diagnostic(base, view, y_lab, classification, seed):
    """Read weak, ICS and equal-cosine rotations through one supervised probe."""
    estimator = LogisticRegression(max_iter=1000) if classification else Ridge()
    probe = make_pipeline(StandardScaler(), estimator).fit(base["e_lab"], y_lab)
    weak = torch.from_numpy(base["e_unl"])
    generator = torch.Generator().manual_seed(seed)
    weak_readout = probe.predict(weak.numpy())
    weak_teacher = base["t_unl"]
    if classification:
        weak_teacher = weak_teacher.argmax(1)
    values = {"cosine_ics": [], "cosine_angular": [], "drift_ics": [], "drift_angular": [], "drift_teacher": []}
    for strong_np, strong_teacher in zip(view["embedding"], view["teacher"], strict=True):
        strong = torch.from_numpy(strong_np)
        c = F.cosine_similarity(weak, strong, dim=1).clamp(-1, 1)
        valid = (weak.norm(dim=1) > 1e-6) & (strong.norm(dim=1) > 1e-6)
        rotated = rotate(weak, torch.randn(weak.shape, generator=generator), c[:, None])
        measured = F.cosine_similarity(weak, rotated, dim=1)
        if not torch.allclose(c[valid], measured[valid], atol=1e-4, rtol=0):
            raise RuntimeError("Cosine rotasi tidak cocok dengan ICS")
        strong_readout = probe.predict(strong_np)
        angular_readout = probe.predict(rotated.numpy())
        if classification:
            strong_teacher = strong_teacher.argmax(1)
        values["cosine_ics"].append(np.where(valid, c.numpy(), np.nan))
        values["cosine_angular"].append(np.where(valid, measured.numpy(), np.nan))
        values["drift_ics"].append(np.abs(strong_readout - weak_readout) if not classification else strong_readout != weak_readout)
        values["drift_angular"].append(np.abs(angular_readout - weak_readout) if not classification else angular_readout != weak_readout)
        values["drift_teacher"].append(np.abs(strong_teacher - weak_teacher) if not classification else strong_teacher != weak_teacher)
    return {name: np.stack(rows) for name, rows in values.items()}


def train_head(name, task, dataset, levels, y_lab, y_val, base, labeled_views, unlabeled_views, seed, steps, device):
    torch.manual_seed(seed)
    lab_rng = torch.Generator(device=device).manual_seed(seed)
    unl_rng = torch.Generator(device=device).manual_seed(seed + 1)
    classification = task == "classification"
    tensor = lambda x: torch.as_tensor(x, device=device)
    xl, xu, xv, xt = (tensor(base[f"e_{split}"]) for split in ("lab", "unl", "val", "test"))
    vl = tensor(labeled_views) if labeled_views is not None else None
    vu = tensor(unlabeled_views) if unlabeled_views is not None else None
    tu = tensor(base["t_unl"])
    pseudo_donor, donor_conf = None, None
    if name == "head_tabcutmix":
        if classification:
            donor_conf, pseudo_donor = tu.max(1)
        else:
            raw = base["t_unl"]
            bins = (np.abs(raw[:, None] - levels).argmin(1) if levels is not None
                    else np.searchsorted(np.quantile(raw, np.linspace(0.1, 0.9, 9)), raw))
            pseudo_donor, donor_conf = tensor(bins), torch.ones(len(tu), device=device)
    n_out = tu.shape[1] if classification else 1
    target_mean, target_sd = 0.0, 1.0
    if not classification:
        target_mean, target_sd = float(y_lab.mean()), float(y_lab.std() + 1e-6)
        y_lab = (y_lab - target_mean) / target_sd
        tu = (tu - target_mean) / target_sd
    yl = tensor(y_lab)
    body = nn.Sequential(nn.Linear(xl.shape[1], 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, 256), nn.ReLU())
    model = nn.Sequential(body, nn.Linear(256, n_out)).to(device)
    ema = deepcopy(model).eval().requires_grad_(False) if name in MATCH_VARIANTS else None
    latent_op = ("gaussian", 0.5) if name == "head_match_gaussian" else LATENT_DISTILL.get(name)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(x):
        out = model(x)
        return out if classification else out[:, 0]

    @torch.no_grad()
    def predict(x):
        model.eval()
        out = forward(x)
        model.train()
        return (out.softmax(1) if classification else out * target_sd + target_mean).cpu().numpy()

    def supervised_loss(pred, target):
        return F.cross_entropy(pred, target) if classification else F.mse_loss(pred, target)

    def get_view(views, original, idx, include_original=False):
        if views is None:
            return original[idx]
        pick = int(aug_rng.integers(len(views) + include_original))
        return original[idx] if pick == len(views) else views[pick, idx]

    metric, sign = METRIC[dataset]
    best_score, best_state, best_step, bad_checks = -np.inf, None, 0, 0
    use_unlabeled = name not in ("head_sup", "ics_sup")
    aug_rng = np.random.default_rng(seed)
    aug_cos = {"labeled_embedding": [], "unlabeled_embedding": [], "unlabeled_hidden": []}
    match_count = torch.zeros((), device=device)
    best_accept_rate = None
    for step in range(1, steps + 1):
        il = torch.randint(len(xl), (min(32, len(xl)),), device=device, generator=lab_rng)
        labeled = get_view(vl, xl, il, include_original=True)
        loss = supervised_loss(forward(labeled), yl[il])
        if vl is not None:
            aug_cos["labeled_embedding"].append(F.cosine_similarity(xl[il], labeled).mean())
        if use_unlabeled:
            iu = torch.randint(len(xu), (224,), device=device, generator=unl_rng)
            strong = get_view(vu, xu, iu)
            if vu is not None:
                aug_cos["unlabeled_embedding"].append(F.cosine_similarity(xu[iu], strong).mean())
            hidden = body(strong)
            if latent_op:
                op, strength = latent_op
                donor = (pseudo_donor[iu], donor_conf[iu]) if pseudo_donor is not None else ()
                augmented = perturb(op, hidden[:, None], strength, *donor)[:, 0]
                with torch.no_grad():
                    aug_cos["unlabeled_hidden"].append(F.cosine_similarity(hidden, augmented).mean())
            else:
                augmented = hidden
            out = model[1](augmented)
            if ema is not None:
                with torch.no_grad():
                    weak = ema(xu[iu])
                if classification:
                    confidence, pseudo = weak.softmax(1).max(1)
                    mask = confidence >= CONFIDENCE
                    match_count += mask.sum()
                    unsup = (F.cross_entropy(out, pseudo, reduction="none") * mask).mean()
                else:
                    unsup = F.mse_loss(out[:, 0], weak[:, 0])
            elif classification:
                unsup = -(tu[iu] * F.log_softmax(out, dim=1)).sum(1).mean()
            else:
                unsup = F.mse_loss(out[:, 0], tu[iu])
            loss = loss + unsup
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if ema is not None:
            with torch.no_grad():
                for average, current in zip(ema.parameters(), model.parameters(), strict=True):
                    average.lerp_(current, 1 - EMA_DECAY)

        if step % 50 == 0 or step == steps:
            pred = predict(xv)
            if levels is not None:
                cutoffs = calibrate(y_val, pred, levels)
                score = cohen_kappa_score(y_val, levels[np.searchsorted(cutoffs, pred, side="right")],
                                          labels=levels, weights="quadratic")
            else:
                score = scores(task, y_val, pred.argmax(1) if classification else pred)[metric]
            if np.isfinite(score) and sign * score > best_score:
                best_score, best_step, bad_checks = sign * score, step, 0
                best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
                if ema is not None and classification:
                    best_accept_rate = float(match_count / (224 * step))
            else:
                bad_checks += 1
                if bad_checks >= 20:
                    break
    if best_state is None:
        raise RuntimeError(f"Semua skor validation tidak finite untuk {dataset} {name}")
    model.load_state_dict(best_state)
    info = {"best_step": best_step, "steps_run": step,
            "cosine": {key: float(torch.stack(vals).mean()) for key, vals in aug_cos.items() if vals}}
    if ema is not None and classification:
        info["unlabeled_accept_rate"] = float(match_count / (224 * step))
        info["unlabeled_accept_rate_through_checkpoint"] = best_accept_rate
    return predict(xv), predict(xt), info


def save_json(path, value):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(value, allow_nan=False, indent=2) + "\n")
    tmp.replace(path)


def read_jsonl(path, repair_tail=False):
    if not path.exists():
        return []
    lines = path.read_text().splitlines(keepends=True)
    rows = []
    for i, line in enumerate(lines):
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            if i != len(lines) - 1 or line.endswith("\n"):
                raise ValueError(f"JSONL rusak: {path}:{i + 1}") from None
            if repair_tail:
                tmp = path.with_suffix(".tmp")
                tmp.write_text("".join(lines[:i]))
                tmp.replace(path)
        else:
            if repair_tail and not line.endswith("\n"):
                with path.open("a") as sink:
                    sink.write("\n")
    return rows


def group_key(dataset, percent, seed):
    return f"{dataset}_{percent:02d}_s{seed}"


def digest(path):
    with path.open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()


def split_files(root):
    return [root / f"label_{p:02d}" / dataset / file
            for p in PERCENTS for dataset in DATASETS
            for file in ("manifest.json", *(f"{split}.npz" for split in SPLITS))]


def run_group(args, profile, dataset, percent, seed, result_path, done):
    names = eligible(args.names, percent)
    task, splits, levels = read_data(args.data, dataset, percent)
    fingerprint = {
        "settings": args.config,
        "group": [dataset, percent, seed],
    }
    key = group_key(dataset, percent, seed)
    signature = hashlib.sha256(json.dumps(fingerprint, sort_keys=True).encode()).hexdigest()[:16]
    folder = profile / "cache" / f"{key}_{signature}"
    todo = [
        name for name in names
        if (key, signature, name) not in done or
        not (profile / "prediksi" / key / f"{name}.npz").is_file()
    ]
    if not todo:
        return
    indices = choose_unlabeled(len(splits["train_unlabeled"][0]), args.max_unlabeled, seed)
    new_model = lambda: make_model(task, seed, args.weights, args.device)
    base = cached(folder / "base.npz", lambda: build_base(new_model, task, splits, indices, seed, args.chunk))
    required = {"ics_distill_90": "90", "ics_distill_70": "70", "ics_distill_50": "50", "ics_distill_ta": "ta",
                "ics_match_70": "70", "ics_both_70": "70"}
    settings = {required[name] for name in names if name in required}
    if "ics_sup" in names or "ics_both_70" in names:
        settings.add("lab70")
    views, diagnostic_paths = {}, {}
    prep_seconds = float(base["build_seconds"])
    y_lab = splits["train_labeled"][1]
    for setting in sorted(settings):
        view = cached(folder / f"ics_{setting}.npz", lambda: build_views(
            new_model, task, splits, indices, seed, args.chunk, args.views, setting
        ))
        prep_seconds += float(view["build_seconds"])
        views[setting] = view["embedding"]
        if setting != "lab70":
            path = profile / "diagnostik" / key / f"ics_{setting}.npz"
            if not path.exists():
                diagnostic_arrays = diagnostic(base, view, y_lab, task == "classification", seed)
                atomic_npz(path, {
                    **diagnostic_arrays, "context_sizes": view["context_sizes"], "rho": view["rho"],
                    "unlabeled_rows": indices,
                }, compress=True)
            diagnostic_paths[setting] = str(path.relative_to(profile))
    y_val, y_test = splits["validation"][1], splits["test"][1]
    for name in todo:
        start = time.monotonic()
        if name == "tabpfn":
            pv, pt, info = base["t_val"], base["t_test"], {"best_step": None}
        else:
            lv = views["lab70"] if name in ("ics_sup", "ics_both_70") else None
            uv = views.get(required.get(name))
            pv, pt, info = train_head(name, task, dataset, levels, y_lab, y_val, base, lv, uv, seed, args.steps, args.device)
        result, arrays = evaluate(task, levels, y_val, y_test, pv, pt)
        pred_path = profile / "prediksi" / key / f"{name}.npz"
        atomic_npz(pred_path, arrays, compress=True)
        row = {
            "name": name, "dataset": dataset, "label_percent": percent, "seed": seed, "task": task,
            "ssl_method": ("label_only" if name in SUPERVISED else
                           "ema_self_training" if name in MATCH_VARIANTS else "tabpfn_distillation"),
            **result, **info, "seconds": time.monotonic() - start, "tabpfn_seconds": prep_seconds,
            "tabpfn_version": version("tabpfn"), "weights": WEIGHT_NAME, "weights_sha256": args.weights_sha256,
            "data_and_config": signature, "n_unlabeled": len(indices), "predictions": str(pred_path.relative_to(profile)),
            "diagnostic": diagnostic_paths.get(required.get(name)),
        }
        with result_path.open("a", encoding="utf-8") as sink:
            sink.write(json.dumps(row, allow_nan=False) + "\n")
            sink.flush()
            os.fsync(sink.fileno())
        metric, _ = METRIC[dataset]
        print(f"{key} {name:15s} val {result['validation'][metric]:.4f} test {result['test'][metric]:.4f}", flush=True)


def partition(groups, count):
    # Stage 7's measured cost makes the three high-context Superconductivity groups dominate runtime.
    cost = {"nhanes": {1: 12, 10: 21, 30: 27}, "pendigits": {1: 8, 10: 13, 30: 18},
            "superconductivity": {1: 80, 10: 225, 30: 715}}
    batches, totals = [[] for _ in range(count)], [0] * count
    for group in sorted(groups, key=lambda group: -cost[group[0]][group[1]]):
        i = totals.index(min(totals))
        batches[i].append(group)
        totals[i] += cost[group[0]][group[1]]
    return [sorted(batch, key=groups.index) for batch in batches]


def parse_list(text, allowed):
    values = list(dict.fromkeys(text.split(",")))
    lookup = {str(value): value for value in allowed}
    if any(value not in lookup for value in values):
        raise SystemExit(f"Pilihan {text!r} tidak valid. Pilih dari {', '.join(lookup)}")
    return [lookup[value] for value in values]


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", default=os.getenv("DATA_DIR"))
    parser.add_argument("--out", type=Path, default=Path("/kaggle/working/tahap8"))
    parser.add_argument("--weights", default=os.getenv("TABPFN_WEIGHTS"))
    parser.add_argument("--device", choices=("cpu", "cuda"), default="cuda")
    parser.add_argument("--gpus", type=int)
    parser.add_argument("--worker", type=int, help=argparse.SUPPRESS)
    parser.add_argument("--smoke", action="store_true")
    parser.add_argument("--steps", type=int)
    parser.add_argument("--views", type=int)
    parser.add_argument("--chunk", type=int, default=2000)
    parser.add_argument("--max-unlabeled", type=int, help="0 = semua baris")
    parser.add_argument("--datasets", default=",".join(DATASETS))
    parser.add_argument("--percents", default=",".join(map(str, PERCENTS)))
    parser.add_argument("--seeds", default=",".join(map(str, SEEDS)))
    parser.add_argument("--names", default="core", help="core, all, atau nama varian dipisah koma")
    args = parser.parse_args()
    args.names = experiment_names(args.names)
    datasets = parse_list(args.datasets, DATASETS)
    percents = parse_list(args.percents, PERCENTS)
    seeds = parse_list(args.seeds, SEEDS)
    default_steps, default_views, default_limit = (200, 2, 512) if args.smoke else (3000, 4, 5000)
    args.steps = default_steps if args.steps is None else args.steps
    args.views = default_views if args.views is None else args.views
    args.max_unlabeled = default_limit if args.max_unlabeled is None else args.max_unlabeled
    if min(args.steps, args.views, args.chunk) < 1 or args.max_unlabeled < 0:
        parser.error("steps, views, chunk harus positif, max-unlabeled minimal 0")
    args.data = find_data(args.data)
    args.out = args.out.resolve()
    profile = args.out / ("smoke" if args.smoke else "full")
    profile.mkdir(parents=True, exist_ok=True)
    groups = [(d, p, s) for p in percents for d in datasets for s in seeds if eligible(args.names, p)]
    if args.smoke:
        smoke_groups = {("nhanes", 10, 42), ("pendigits", 1, 42), ("superconductivity", 30, 42)}
        groups = [group for group in groups if group in smoke_groups]
    if not groups:
        parser.error("Tidak ada grup yang cocok dengan pilihan ini")

    visible = os.getenv("CUDA_VISIBLE_DEVICES")
    if args.worker is None:
        available = torch.cuda.device_count() if args.device == "cuda" else 1
        count = args.gpus if args.gpus is not None else min(available, 1 if args.smoke else 2)
        if not 1 <= count <= available:
            parser.error(f"Diminta {count} worker, perangkat tersedia {available}. Aktifkan GPU Kaggle atau pakai --device cpu.")
    else:
        count = args.gpus
        if args.worker not in range(count):
            parser.error("worker di luar rentang")

    if version("tabpfn") != TABPFN_VERSION:
        parser.error(f"Pakai tabpfn=={TABPFN_VERSION} agar sebanding dengan tahap 7 v3.5")
    args.weights = weights(args.out, args.weights)
    args.weights_sha256 = digest(args.weights)
    data_sha256 = hashlib.sha256("".join(digest(path) for path in split_files(args.data)).encode()).hexdigest()
    args.config = {
        "schema": SCHEMA, "steps": args.steps, "views": args.views, "max_unlabeled": args.max_unlabeled,
        "chunk": args.chunk, "device": args.device, "tabpfn_version": TABPFN_VERSION,
        "weights_sha256": args.weights_sha256, "data_sha256": data_sha256,
        "ema_decay": EMA_DECAY, "pseudolabel_confidence": CONFIDENCE,
    }
    if args.worker is None:
        with (profile / ".run.lock").open("w") as lock:
            try:
                fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
            except BlockingIOError:
                parser.error(f"Training lain masih menulis ke {profile}")
            config_path = profile / "config.json"
            if config_path.exists() and json.loads(config_path.read_text()) != args.config:
                parser.error("Konfigurasi pada folder hasil sudah berbeda. Gunakan --out baru.")
            save_json(config_path, args.config)
            for path in profile.glob("results_gpu*.jsonl"):
                read_jsonl(path, repair_tail=True)
            gpu_ids = visible.split(",") if visible else list(map(str, range(count)))
            if count == 1:
                run_worker(args, profile, groups, 0)
                return
            procs = [subprocess.Popen(
                [sys.executable, __file__, *sys.argv[1:], "--worker", str(i), "--gpus", str(count),
                 "--weights", str(args.weights)],
                env={**os.environ, "CUDA_VISIBLE_DEVICES": gpu_ids[i]},
            ) for i in range(count)]
            try:
                codes = [proc.wait() for proc in procs]
            finally:
                for proc in procs:
                    if proc.poll() is None:
                        proc.terminate()
                        proc.wait()
            if any(codes):
                raise SystemExit(f"Worker gagal dengan exit code {codes}")
    else:
        run_worker(args, profile, partition(groups, count)[args.worker], args.worker)


def run_worker(args, profile, groups, worker):
    done = {(group_key(row["dataset"], row["label_percent"], row["seed"]), row["data_and_config"], row["name"])
            for path in profile.glob("results_gpu*.jsonl") for row in read_jsonl(path)}
    result_path = profile / f"results_gpu{worker}.jsonl"
    print(f"Worker {worker}: {len(groups)} grup, {sum(len(eligible(args.names, p)) for _, p, _ in groups)} run", flush=True)
    for dataset, percent, seed in groups:
        run_group(args, profile, dataset, percent, seed, result_path, done)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/ringkas.py
"""Ringkas JSONL dan diagnostik tahap 8 di Kaggle atau dari hasil unduhan."""

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd

from tabpfn_konteks import (
    CORE, DATASETS, LATENT_DISTILL, MATCH_VARIANTS, METRIC, PERCENTS, SEEDS, SSL_AUGMENTED, SSL_IDENTITY, SUPERVISED, VARIANTS,
    eligible, experiment_names, read_jsonl,
)

KEY = ["dataset", "label_percent", "seed", "name"]
COSINE_BINS = np.linspace(-1, 1, 41)
FAMILIES = {
    "Supervised / label-only": SUPERVISED,
    "SSL tanpa augmentasi eksplisit": SSL_IDENTITY,
    "SSL dengan augmentasi": SSL_AUGMENTED,
    "SSL pilihan validation": (*SSL_IDENTITY, *SSL_AUGMENTED),
}
EFFECTS = (
    ("Distilasi SSL", "head_distill", "head_sup"),
    ("Self-training EMA", "head_match", "head_sup"),
    ("Distilasi vs EMA, view identik", "head_distill", "head_match"),
    ("ICS pada distilasi", "ics_distill_70", "head_distill"),
    ("ICS pada EMA", "ics_match_70", "head_match"),
    ("Gaussian pada EMA", "head_match_gaussian", "head_match"),
    ("Distilasi vs EMA, keduanya Gaussian", "head_gaussian", "head_match_gaussian"),
    ("ICS vs Gaussian pada EMA", "ics_match_70", "head_match_gaussian"),
    ("Distilasi vs EMA, keduanya ICS", "ics_distill_70", "ics_match_70"),
    ("ICS pada supervised", "ics_sup", "head_sup"),
    ("Tambah SSL pada ICS supervised", "ics_both_70", "ics_sup"),
    ("Tambah ICS labeled pada ICS-distill", "ics_both_70", "ics_distill_70"),
    ("ICS vs Gaussian pada distilasi", "ics_distill_70", "head_gaussian"),
    ("ICS vs angular pada distilasi", "ics_distill_70", "head_angular"),
    ("ICS vs SCARF pada distilasi", "ics_distill_70", "head_scarf"),
)


def table(columns, rows):
    yield "| " + " | ".join(columns) + " |"
    yield "|" + "|".join("---" for _ in columns) + "|"
    for row in rows:
        yield "| " + " | ".join(map(str, row)) + " |"


def verify(profile, rows, expected, allowed):
    observed = {tuple(row[key] for key in KEY) for row in rows}
    if expected - observed or observed - allowed:
        raise ValueError(f"Grid tidak sesuai: hilang={sorted(expected - observed)}, ekstra={sorted(observed - allowed)}")
    for row in rows:
        if row["task"] == "classification" and row["name"] in MATCH_VARIANTS:
            for key in ("unlabeled_accept_rate", "unlabeled_accept_rate_through_checkpoint"):
                if row.get(key) is None or not 0 <= row[key] <= 1:
                    raise ValueError(f"Acceptance rate EMA tidak valid: {row['predictions']}")
        with np.load(profile / row["predictions"], allow_pickle=False) as arrays:
            for split in ("validation", "test"):
                if len(arrays[f"pred_{split}"]) != len(arrays[f"y_{split}"]):
                    raise ValueError(f"Shape prediksi tidak cocok: {row['predictions']}")
                if row["task"] == "ordinal_regression" and f"pred_{split}_calibrated" not in arrays:
                    raise ValueError(f"QWK calibrated tidak tersedia: {row['predictions']}")
        if row["diagnostic"]:
            with np.load(profile / row["diagnostic"], allow_pickle=False) as diag:
                if not np.allclose(diag["cosine_ics"], diag["cosine_angular"], atol=1e-4, rtol=0, equal_nan=True):
                    raise ValueError(f"Cosine pembanding tidak cocok: {row['diagnostic']}")


def summarize_diagnostic(profile, rows):
    path_groups = {row["diagnostic"]: (row["dataset"], row["label_percent"], row["seed"]) for row in rows if row["diagnostic"]}
    bins = []
    group_means = []
    for path, group in sorted(path_groups.items()):
        dataset, percent, seed = group
        rho = Path(path).stem.removeprefix("ics_")
        with np.load(profile / path, allow_pickle=False) as data:
            keep = np.isfinite(data["cosine_ics"])
            cos = data["cosine_ics"][keep]
            drift = {key: data[f"drift_{key}"][keep] for key in ("ics", "angular", "teacher")}
        if not len(cos):
            continue
        group_means.append((dataset, percent, seed, rho, len(cos), cos.mean(),
                            *(arr.mean() for arr in drift.values())))
        idx = np.clip(np.searchsorted(COSINE_BINS, cos, side="right") - 1, 0, len(COSINE_BINS) - 2)
        for i in np.unique(idx):
            mask = idx == i
            bins.append((dataset, percent, seed, rho, COSINE_BINS[i], COSINE_BINS[i + 1], int(mask.sum()),
                         cos[mask].mean(), *(arr[mask].mean() for arr in drift.values())))
    if not group_means:
        return []
    cols = ["dataset", "label_percent", "seed", "rho"]
    metrics = ["n", "cosine", "ics", "angular", "teacher"]
    group_df = pd.DataFrame(group_means, columns=[*cols, *metrics])
    group_df.to_csv(profile / "diagnostik_per_grup.csv", index=False)
    bin_df = pd.DataFrame(bins, columns=[*cols, "cos_min", "cos_max", *metrics])
    bin_df.to_csv(profile / "diagnostik_cosine.csv", index=False)

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    primary = bin_df[bin_df.rho == "70"]
    if not primary.empty:
        fig, axes = plt.subplots(3, 3, figsize=(11, 8), squeeze=False, layout="constrained")
        for i, dataset in enumerate(DATASETS):
            for j, percent in enumerate(PERCENTS):
                ax = axes[i, j]
                subset = primary[(primary.dataset == dataset) & (primary.label_percent == percent)]
                agg = subset.groupby("cos_min")[["cosine", "ics", "angular"]].mean()
                ax.set_title(f"{dataset} · {percent}%")
                if j == 0:
                    ax.set_ylabel("Fraksi kelas berubah" if dataset == "pendigits" else "Rerata |Δ prediksi|")
                if i == 2:
                    ax.set_xlabel("Cosine ke weak")
                if agg.empty:
                    ax.set_axis_off()
                    continue
                for method, label in (("ics", "ICS"), ("angular", "Angular")):
                    ax.plot(agg.cosine, agg[method], marker="o", markersize=3, label=label)
                ax.legend()
        fig.savefig(profile / "diagnostik_cosine.png", dpi=160)
        plt.close(fig)

    lines = [
        "## Diagnostik H1 pada ρ = 0,7", "",
        "Rerata tiap seed diberi bobot sama. Kolom ICS dan angular memakai probe yang sama. "
        "Drift teacher berasal dari prediksi TabPFN penuh dan subset, sehingga dilaporkan sebagai pengukuran terpisah.",
        "",
    ]
    grouped = group_df[group_df.rho == "70"].groupby(["dataset", "label_percent"])
    pairs = [[dataset, percent, len(block), *(f"{block[key].mean():.5f}" for key in ("cosine", "ics", "angular", "teacher"))]
             for (dataset, percent), block in grouped]
    lines.extend(table(["Dataset", "Label %", "Seed", "Cosine", "Drift ICS", "Drift angular", "Drift teacher"], pairs))
    lines += ["", "Data per seed dan bin cosine ada di `diagnostik_per_grup.csv` dan `diagnostik_cosine.csv`.", ""]
    return lines


def validation_selection(frame):
    result = []
    for (dataset, percent), cell in frame.groupby(["dataset", "label_percent"], sort=False):
        metric, sign = METRIC[dataset]
        column = f"validation_{metric}"
        blocks = {name: rows for name, rows in cell.groupby("name")}
        if not all(name in blocks and tuple(sorted(blocks[name].seed)) == SEEDS
                   and blocks[name][column].notna().all() for name in CORE):
            continue
        teacher_score = blocks["tabpfn"][column].mean()
        for family, methods in FAMILIES.items():
            winner = max(methods, key=lambda name: sign * blocks[name][column].mean())
            rows = blocks[winner]
            validation = rows[column].mean()
            result.append({
                "dataset": dataset, "label_percent": percent, "family": family, "name": winner, "metric": metric,
                "validation": validation, "test": rows[f"test_{metric}"].mean(),
                "validation_gain_vs_tabpfn": sign * (validation - teacher_score),
            })
    return result


def compare_history(profile, frame, selected):
    history_path = Path(__file__).with_name("stage8_history.json")
    config_path = profile / "config.json"
    lines = ["## Pembanding historis", ""]
    if not history_path.is_file() or not config_path.is_file():
        return lines + ["Metadata referensi atau konfigurasi tidak tersedia.", ""]
    history = json.loads(history_path.read_text())
    if json.loads(config_path.read_text())["data_sha256"] != history["data_sha256"]:
        return lines + ["Fingerprint split berbeda dari referensi repo. Tabel historis dilewati.", ""]
    rows = history["rows"]
    bridge = []
    for winner in selected:
        dataset, percent, name = (winner[key] for key in ("dataset", "label_percent", "name"))
        metric = "qwk" if dataset == "nhanes" else METRIC[dataset][0]
        current = frame[(frame.dataset == dataset) & (frame.label_percent == percent) & (frame.name == name)]
        references = [r for r in rows if (r["dataset"], r["label_percent"]) == (dataset, percent)]
        for reference in references:
            bridge.append({
                "dataset": dataset, "label_percent": percent, "metric": metric, "family": winner["family"],
                "stage8_name": name, "stage8_validation": current[f"validation_{metric}"].mean(),
                "stage8_test": current[f"test_{metric}"].mean(), "reference_series": reference["series"],
                "reference_name": reference["name"], "reference_validation": reference["validation"],
                "reference_test": reference["test"], "reference_picked_by_validation": reference["picked_by_validation"],
            })
    pd.DataFrame(bridge).to_csv(profile / "pembanding_historis.csv", index=False)
    lines += [
        "CSV memuat supervised dan RankUp tahap 5, RankUp dan SSR-RCUS tahap 6, serta teacher dan distill tahap 7 versi 2.5/3.5. "
        "Varian historis terpilih memakai rerata validation pada setiap sel. Arsitektur dan protokol checkpoint mengikuti tahap asal. "
        "Untuk NHANES, CSV memakai QWK mentah pada kedua sisi. Pemilihan tahap 8 tetap memakai QWK terkalibrasi.",
        "",
    ]
    compact = []
    for dataset in DATASETS:
        for percent in PERCENTS:
            matches = [r for r in bridge if (r["dataset"], r["label_percent"], r["family"])
                       == (dataset, percent, "SSL pilihan validation")]
            if not matches:
                continue
            stage8 = matches[0]
            anchors = ["Tahap 5 · TabM", "Tahap 6 · TabM", "Tahap 7 · 2.5", "Tahap 7 · 3.5"]
            refs = {r["reference_series"]: f"{r['reference_name']} ({r['reference_test']:.4f})"
                    for r in matches if r["reference_picked_by_validation"]}
            compact.append([dataset, percent, stage8["metric"], f"{stage8['stage8_name']} ({stage8['stage8_test']:.4f})",
                            *(refs.get(source, "—") for source in anchors)])
    lines.extend(table(
        ["Dataset", "Label %", "Metrik test", "SSL tahap 8", "SSL tahap 5", "RCUS tahap 6",
         "Augmentasi tahap 7 · 2.5", "Augmentasi tahap 7 · 3.5"], compact,
    ))
    lines.append("")
    return lines


def report(profile, rows):
    wide_rows = []
    for row in rows:
        flattened = {k: row[k] for k in KEY}
        for key in ("ssl_method", "unlabeled_accept_rate", "unlabeled_accept_rate_through_checkpoint"):
            flattened[key] = row.get(key)
        for split in ("validation", "test"):
            flattened.update({f"{split}_{k}": v for k, v in row[split].items()})
        wide_rows.append(flattened)
    frame = pd.DataFrame(wide_rows)
    frame.to_csv(profile / "skor_per_run.csv", index=False)
    lines = ["# Ringkasan augmentasi konteks", "", f"{len(rows)} run selesai.", "",
             "NHANES memakai QWK terkalibrasi untuk pemilihan checkpoint. Batas kelas hanya dari validation. "
             "QWK mentah, Spearman dan MSE tetap dicatat. Angka test hanya dilaporkan.", ""]
    chosen = validation_selection(frame)
    if chosen:
        pd.DataFrame(chosen).to_csv(profile / "pilihan_validation.csv", index=False)
        lines += ["## Pilihan per dataset dan fraksi label", "",
                  "Setiap sel memerlukan grid utama lengkap pada ketiga seed. Metode dipilih dari rerata validation "
                  "untuk masing-masing keluarga. Δ positif berarti membaik terhadap teacher TabPFN di validation.", ""]
        lines.extend(table(
            ["Dataset", "Label %", "Keluarga", "Pilihan", "Validation", "Test", "Δ validation vs teacher"],
            ([r["dataset"], r["label_percent"], r["family"], r["name"], f"{r['validation']:.4f}",
              f"{r['test']:.4f}", f"{r['validation_gain_vs_tabpfn']:+.4f}"] for r in chosen),
        ))
        lines.append("")
        lines.extend(compare_history(profile, frame, chosen))
    for dataset, (metric, sign) in METRIC.items():
        data = frame[frame.dataset == dataset]
        if data.empty:
            continue
        lines += [f"## {dataset} ({metric})", ""]
        summaries = []
        for name in VARIANTS:
            subset = data[data.name == name]
            if subset.empty:
                continue
            for percent, group in subset.groupby("label_percent"):
                def mean_std(column):
                    vals = group[column].dropna()
                    return f"{vals.mean():.4f} ± {vals.std():.4f}" if len(vals) > 1 else f"{vals.mean():.4f}"
                summaries.append([name, percent, len(group), mean_std(f"validation_{metric}"), mean_std(f"test_{metric}")])
        lines.extend(table(["Varian", "Label %", "Seed", "Validation", "Test"], summaries))
        lines += [""]
        val = data.pivot(index=["label_percent", "seed"], columns="name", values=f"validation_{metric}")
        comparisons = []
        pairs = [*EFFECTS, *((f"Efek {name}", name, "head_distill") for name in
                            (*LATENT_DISTILL, "ics_distill_90", "ics_distill_50", "ics_distill_ta"))]
        for effect, variant, base in pairs:
            if variant not in val.columns or base not in val.columns:
                continue
            for percent, group in val.groupby(level="label_percent"):
                delta = (sign * (group[variant] - group[base])).dropna()
                if len(delta):
                    comparisons.append([effect, variant, base, percent, f"{(delta > 0).sum()}/{len(delta)}", f"{delta.mean():+.4f}"])
        if comparisons:
            lines.extend(table(["Pertanyaan", "Varian", "Pembanding", "Label %", "Menang validation", "Δ rerata searah perbaikan"], comparisons))
            lines.append("")
    lines.extend(summarize_diagnostic(profile, rows))
    (profile / "ringkasan.md").write_text("\n".join(lines) + "\n")
    print(f"{profile}: {len(rows)} run, ringkasan.md dan skor_per_run.csv", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("profile", type=Path)
    parser.add_argument("--expect", choices=("smoke", "core", "all"))
    args = parser.parse_args()
    unique = {}
    for path in sorted(args.profile.glob("results_gpu*.jsonl")):
        for row in read_jsonl(path):
            key = tuple(row[name] for name in KEY)
            previous = unique.get(key)
            if previous and previous["data_and_config"] != row["data_and_config"]:
                raise ValueError(f"Konfigurasi campur untuk {key}")
            unique[key] = row
    rows = list(unique.values())
    if not rows:
        raise SystemExit(f"Belum ada hasil di {args.profile}")
    if args.expect:
        groups = [("nhanes", 10, 42), ("pendigits", 1, 42), ("superconductivity", 30, 42)] if args.expect == "smoke" \
            else [(d, p, s) for d in DATASETS for p in PERCENTS for s in SEEDS]
        names = CORE if args.expect == "smoke" else experiment_names(args.expect)
        expected = {(d, p, s, name) for d, p, s in groups for name in eligible(names, p)}
        allowed = ({(d, p, s, name) for d, p, s in groups for name in eligible(VARIANTS, p)}
                   if args.expect == "core" else expected)
        verify(args.profile, rows, expected, allowed)
    report(args.profile, rows)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/stage8_history.json
{"data_sha256":"d6724d3a853d0076fd5f8716493267382159616675b436865be2d8d77515cff6","rows":[{"dataset":"nhanes","label_percent":1,"series":"Tahap 5 · TabM","name":"supervised","picked_by_validation":false,"validation":0.13417326372616611,"test":0.16899973311801764},{"dataset":"nhanes","label_percent":1,"series":"Tahap 5 · TabM","name":"rankup_noaug","picked_by_validation":false,"validation":0.2244992234759392,"test":0.2320764647465663},{"dataset":"nhanes","label_percent":1,"series":"Tahap 5 · TabM","name":"both_scarf","picked_by_validation":true,"validation":0.3222498611621422,"test":0.28003300923355035},{"dataset":"nhanes","label_percent":10,"series":"Tahap 5 · TabM","name":"supervised","picked_by_validation":false,"validation":0.2950125822932838,"test":0.2540797385469509},{"dataset":"nhanes","label_percent":10,"series":"Tahap 5 · TabM","name":"rankup_noaug","picked_by_validation":false,"validation":0.29630578109514333,"test":0.2463744171048746},{"dataset":"nhanes","label_percent":10,"series":"Tahap 5 · TabM","name":"angular","picked_by_validation":true,"validation":0.3043113669756078,"test":0.2615889107040824},{"dataset":"nhanes","label_percent":30,"series":"Tahap 5 · TabM","name":"supervised","picked_by_validation":false,"validation":0.3262284785558712,"test":0.3107915666624415},{"dataset":"nhanes","label_percent":30,"series":"Tahap 5 · TabM","name":"rankup_noaug","picked_by_validation":false,"validation":0.3254353822897861,"test":0.3052605621781272},{"dataset":"nhanes","label_percent":30,"series":"Tahap 5 · TabM","name":"both_zero_out","picked_by_validation":true,"validation":0.3266901672351319,"test":0.31307896872581814},{"dataset":"pendigits","label_percent":1,"series":"Tahap 5 · TabM","name":"supervised","picked_by_validation":false,"validation":0.8977260149423215,"test":0.8371718292405771},{"dataset":"pendigits","label_percent":1,"series":"Tahap 5 · TabM","name":"rankup_noaug","picked_by_validation":false,"validation":0.9319591898114925,"test":0.8808538451106869},{"dataset":"pendigits","label_percent":1,"series":"Tahap 5 · TabM","name":"matched_gaussian","picked_by_validation":true,"validation":0.9475824650268202,"test":0.8926771011943063},{"dataset":"pendigits","label_percent":10,"series":"Tahap 5 · TabM","name":"supervised","picked_by_validation":false,"validation":0.9873857030611185,"test":0.9495242947328367},{"dataset":"pendigits","label_percent":10,"series":"Tahap 5 · TabM","name":"rankup_noaug","picked_by_validation":true,"validation":0.9884940961989365,"test":0.9492446486132804},{"dataset":"pendigits","label_percent":30,"series":"Tahap 5 · TabM","name":"supervised","picked_by_validation":false,"validation":0.9949205060868539,"test":0.9692864194033768},{"dataset":"pendigits","label_percent":30,"series":"Tahap 5 · TabM","name":"rankup_noaug","picked_by_validation":false,"validation":0.9964906186725728,"test":0.9737283197109958},{"dataset":"pendigits","label_percent":30,"series":"Tahap 5 · TabM","name":"latent_fnoise","picked_by_validation":true,"validation":0.9969288699510149,"test":0.9755856523815577},{"dataset":"superconductivity","label_percent":1,"series":"Tahap 5 · TabM","name":"supervised","picked_by_validation":false,"validation":340.8263537006003,"test":380.5455785487286},{"dataset":"superconductivity","label_percent":1,"series":"Tahap 5 · TabM","name":"rankup_noaug","picked_by_validation":false,"validation":337.70202793417997,"test":366.5114124482035},{"dataset":"superconductivity","label_percent":1,"series":"Tahap 5 · TabM","name":"matched_gaussian","picked_by_validation":true,"validation":334.2698263289673,"test":364.53648509877576},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 5 · TabM","name":"supervised","picked_by_validation":false,"validation":189.39518407545464,"test":207.70105019632837},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 5 · TabM","name":"rankup_noaug","picked_by_validation":false,"validation":190.17013332125748,"test":208.8466922916999},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 5 · TabM","name":"angular_weak_prerelu","picked_by_validation":true,"validation":187.19116009355463,"test":202.293246948442},{"dataset":"superconductivity","label_percent":30,"series":"Tahap 5 · TabM","name":"supervised","picked_by_validation":false,"validation":139.5675721200219,"test":151.92230364051952},{"dataset":"superconductivity","label_percent":30,"series":"Tahap 5 · TabM","name":"rankup_noaug","picked_by_validation":false,"validation":141.17241745886733,"test":152.75514350368914},{"dataset":"superconductivity","label_percent":30,"series":"Tahap 5 · TabM","name":"latent_gaussian","picked_by_validation":true,"validation":137.9890860427044,"test":145.72965232093287},{"dataset":"nhanes","label_percent":1,"series":"Tahap 6 · TabM","name":"rankup","picked_by_validation":false,"validation":0.13757281044959324,"test":0.15783168309560824},{"dataset":"nhanes","label_percent":1,"series":"Tahap 6 · TabM","name":"rcus","picked_by_validation":false,"validation":0.12740624741670456,"test":0.13680011611369872},{"dataset":"nhanes","label_percent":1,"series":"Tahap 6 · TabM","name":"rcus_input_scarf","picked_by_validation":true,"validation":0.1565740130422051,"test":0.16240614578491283},{"dataset":"nhanes","label_percent":10,"series":"Tahap 6 · TabM","name":"rankup","picked_by_validation":false,"validation":0.25771430662106637,"test":0.2087475305006763},{"dataset":"nhanes","label_percent":10,"series":"Tahap 6 · TabM","name":"rcus","picked_by_validation":true,"validation":0.2505273230439383,"test":0.21782208099802283},{"dataset":"nhanes","label_percent":30,"series":"Tahap 6 · TabM","name":"rankup","picked_by_validation":false,"validation":0.25675202087426785,"test":0.2705402964705718},{"dataset":"nhanes","label_percent":30,"series":"Tahap 6 · TabM","name":"rcus","picked_by_validation":false,"validation":0.264868757921548,"test":0.2630859713919494},{"dataset":"nhanes","label_percent":30,"series":"Tahap 6 · TabM","name":"rcus_input_scarf","picked_by_validation":true,"validation":0.2700957563525163,"test":0.2849678300963197},{"dataset":"pendigits","label_percent":1,"series":"Tahap 6 · TabM","name":"rankup","picked_by_validation":false,"validation":0.9044060726966225,"test":0.8424280287735625},{"dataset":"pendigits","label_percent":1,"series":"Tahap 6 · TabM","name":"rcus","picked_by_validation":false,"validation":0.9311106691176259,"test":0.8845468558894188},{"dataset":"pendigits","label_percent":1,"series":"Tahap 6 · TabM","name":"rcus_input_zero_out","picked_by_validation":true,"validation":0.9418591100109283,"test":0.8938107982576453},{"dataset":"pendigits","label_percent":10,"series":"Tahap 6 · TabM","name":"rankup","picked_by_validation":false,"validation":0.9827137449389863,"test":0.9440486265191982},{"dataset":"pendigits","label_percent":10,"series":"Tahap 6 · TabM","name":"rcus","picked_by_validation":true,"validation":0.9907422292092694,"test":0.9725754844866631},{"dataset":"pendigits","label_percent":30,"series":"Tahap 6 · TabM","name":"rankup","picked_by_validation":false,"validation":0.9931912128243423,"test":0.971283920966797},{"dataset":"pendigits","label_percent":30,"series":"Tahap 6 · TabM","name":"rcus","picked_by_validation":false,"validation":0.9957845847611156,"test":0.9778445610590029},{"dataset":"pendigits","label_percent":30,"series":"Tahap 6 · TabM","name":"rcus_angular_weak","picked_by_validation":true,"validation":0.9960801172984916,"test":0.9776192942334737},{"dataset":"superconductivity","label_percent":1,"series":"Tahap 6 · TabM","name":"rankup","picked_by_validation":false,"validation":374.87442242931866,"test":392.901334414607},{"dataset":"superconductivity","label_percent":1,"series":"Tahap 6 · TabM","name":"rcus","picked_by_validation":true,"validation":415.9445257674791,"test":446.29150564044863},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 6 · TabM","name":"rankup","picked_by_validation":false,"validation":199.87743358556187,"test":208.1701558316062},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 6 · TabM","name":"rcus","picked_by_validation":false,"validation":201.6383631202681,"test":208.0624591363433},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 6 · TabM","name":"rcus_input_scarf","picked_by_validation":true,"validation":194.2851407181621,"test":204.33626817888114},{"dataset":"superconductivity","label_percent":30,"series":"Tahap 6 · TabM","name":"rankup","picked_by_validation":false,"validation":145.04105524588016,"test":151.70022391791105},{"dataset":"superconductivity","label_percent":30,"series":"Tahap 6 · TabM","name":"rcus","picked_by_validation":true,"validation":154.51382371803092,"test":154.4633925633908},{"dataset":"nhanes","label_percent":1,"series":"Tahap 7 · 2.5","name":"tabpfn","picked_by_validation":false,"validation":0.07718673288198925,"test":0.08940329254346069},{"dataset":"nhanes","label_percent":1,"series":"Tahap 7 · 2.5","name":"head_distill","picked_by_validation":false,"validation":0.1368343359419588,"test":0.14289483935148595},{"dataset":"nhanes","label_percent":1,"series":"Tahap 7 · 2.5","name":"head_angular","picked_by_validation":true,"validation":0.20935455867379574,"test":0.23687079879058845},{"dataset":"nhanes","label_percent":10,"series":"Tahap 7 · 2.5","name":"tabpfn","picked_by_validation":false,"validation":0.2236415168579405,"test":0.21603484373554363},{"dataset":"nhanes","label_percent":10,"series":"Tahap 7 · 2.5","name":"head_distill","picked_by_validation":false,"validation":0.2569172325289133,"test":0.25465597612468777},{"dataset":"nhanes","label_percent":10,"series":"Tahap 7 · 2.5","name":"head_scarf","picked_by_validation":true,"validation":0.3206635692762488,"test":0.30622192057240954},{"dataset":"nhanes","label_percent":30,"series":"Tahap 7 · 2.5","name":"tabpfn","picked_by_validation":false,"validation":0.3274673234779089,"test":0.3201838910718161},{"dataset":"nhanes","label_percent":30,"series":"Tahap 7 · 2.5","name":"head_distill","picked_by_validation":false,"validation":0.3647563697781427,"test":0.32846459381234405},{"dataset":"nhanes","label_percent":30,"series":"Tahap 7 · 2.5","name":"head_angular","picked_by_validation":true,"validation":0.39842615782595875,"test":0.3714556592557748},{"dataset":"pendigits","label_percent":1,"series":"Tahap 7 · 2.5","name":"tabpfn","picked_by_validation":false,"validation":0.8869870765907402,"test":0.8375370528416075},{"dataset":"pendigits","label_percent":1,"series":"Tahap 7 · 2.5","name":"head_distill","picked_by_validation":false,"validation":0.8909483000657227,"test":0.8388727700563477},{"dataset":"pendigits","label_percent":1,"series":"Tahap 7 · 2.5","name":"head_scarf","picked_by_validation":true,"validation":0.8990329962609753,"test":0.8415973048260843},{"dataset":"pendigits","label_percent":10,"series":"Tahap 7 · 2.5","name":"tabpfn","picked_by_validation":false,"validation":0.9831782119369166,"test":0.9484811498197447},{"dataset":"pendigits","label_percent":10,"series":"Tahap 7 · 2.5","name":"head_distill","picked_by_validation":false,"validation":0.9837802923957342,"test":0.9465295328736281},{"dataset":"pendigits","label_percent":10,"series":"Tahap 7 · 2.5","name":"head_scarf","picked_by_validation":true,"validation":0.9842671261244231,"test":0.944181202119094},{"dataset":"pendigits","label_percent":30,"series":"Tahap 7 · 2.5","name":"tabpfn","picked_by_validation":false,"validation":0.9937774427928804,"test":0.9725726143832892},{"dataset":"pendigits","label_percent":30,"series":"Tahap 7 · 2.5","name":"head_distill","picked_by_validation":false,"validation":0.9944450935288697,"test":0.9693087724840749},{"dataset":"pendigits","label_percent":30,"series":"Tahap 7 · 2.5","name":"head_dropout","picked_by_validation":true,"validation":0.994449756890703,"test":0.9681888143700108},{"dataset":"superconductivity","label_percent":1,"series":"Tahap 7 · 2.5","name":"tabpfn","picked_by_validation":false,"validation":272.45191446940106,"test":306.48028564453125},{"dataset":"superconductivity","label_percent":1,"series":"Tahap 7 · 2.5","name":"head_distill","picked_by_validation":false,"validation":270.76488240559894,"test":303.53424072265625},{"dataset":"superconductivity","label_percent":1,"series":"Tahap 7 · 2.5","name":"head_fnoise","picked_by_validation":true,"validation":269.7834777832031,"test":304.5947265625},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 7 · 2.5","name":"tabpfn","picked_by_validation":false,"validation":158.98221842447916,"test":165.7642364501953},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 7 · 2.5","name":"head_distill","picked_by_validation":false,"validation":159.37648010253906,"test":166.89710998535156},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 7 · 2.5","name":"head_fnoise","picked_by_validation":true,"validation":157.99161783854166,"test":168.08067321777344},{"dataset":"superconductivity","label_percent":30,"series":"Tahap 7 · 2.5","name":"tabpfn","picked_by_validation":false,"validation":112.19809468587239,"test":122.44816589355469},{"dataset":"superconductivity","label_percent":30,"series":"Tahap 7 · 2.5","name":"head_distill","picked_by_validation":false,"validation":111.23772430419922,"test":123.53341929117839},{"dataset":"superconductivity","label_percent":30,"series":"Tahap 7 · 2.5","name":"head_tabcutmix","picked_by_validation":true,"validation":111.37583669026692,"test":123.6783930460612},{"dataset":"nhanes","label_percent":1,"series":"Tahap 7 · 3.5","name":"tabpfn","picked_by_validation":false,"validation":0.08861247577087517,"test":0.08374252232082835},{"dataset":"nhanes","label_percent":1,"series":"Tahap 7 · 3.5","name":"head_distill","picked_by_validation":false,"validation":0.1069364412212771,"test":0.10643328239674954},{"dataset":"nhanes","label_percent":1,"series":"Tahap 7 · 3.5","name":"head_angular","picked_by_validation":true,"validation":0.224073425097395,"test":0.26088145210559577},{"dataset":"nhanes","label_percent":10,"series":"Tahap 7 · 3.5","name":"tabpfn","picked_by_validation":false,"validation":0.23816315638816268,"test":0.25964321256499184},{"dataset":"nhanes","label_percent":10,"series":"Tahap 7 · 3.5","name":"head_distill","picked_by_validation":false,"validation":0.26335216366107234,"test":0.2647366194275858},{"dataset":"nhanes","label_percent":10,"series":"Tahap 7 · 3.5","name":"head_angular","picked_by_validation":true,"validation":0.36238976312892524,"test":0.3453311047970908},{"dataset":"nhanes","label_percent":30,"series":"Tahap 7 · 3.5","name":"tabpfn","picked_by_validation":false,"validation":0.3391316134426135,"test":0.3188726373253399},{"dataset":"nhanes","label_percent":30,"series":"Tahap 7 · 3.5","name":"head_distill","picked_by_validation":false,"validation":0.3534142347741648,"test":0.3209815036998476},{"dataset":"nhanes","label_percent":30,"series":"Tahap 7 · 3.5","name":"head_scarf","picked_by_validation":true,"validation":0.4180139714600249,"test":0.4127987617270003},{"dataset":"pendigits","label_percent":1,"series":"Tahap 7 · 3.5","name":"tabpfn","picked_by_validation":false,"validation":0.9092410053898036,"test":0.8734449423056272},{"dataset":"pendigits","label_percent":1,"series":"Tahap 7 · 3.5","name":"head_distill","picked_by_validation":false,"validation":0.9143221392413534,"test":0.8716546232730386},{"dataset":"pendigits","label_percent":1,"series":"Tahap 7 · 3.5","name":"head_scarf","picked_by_validation":true,"validation":0.916292876365292,"test":0.8742473182755971},{"dataset":"pendigits","label_percent":10,"series":"Tahap 7 · 3.5","name":"tabpfn","picked_by_validation":false,"validation":0.9886534904300651,"test":0.9703005252225638},{"dataset":"pendigits","label_percent":10,"series":"Tahap 7 · 3.5","name":"head_distill","picked_by_validation":false,"validation":0.9898176210331453,"test":0.9672358720018023},{"dataset":"pendigits","label_percent":10,"series":"Tahap 7 · 3.5","name":"head_scarf","picked_by_validation":true,"validation":0.9918414954832434,"test":0.9679789961600084},{"dataset":"pendigits","label_percent":30,"series":"Tahap 7 · 3.5","name":"tabpfn","picked_by_validation":false,"validation":0.9970612549300778,"test":0.9840861007359949},{"dataset":"pendigits","label_percent":30,"series":"Tahap 7 · 3.5","name":"head_distill","picked_by_validation":false,"validation":0.9977556993745222,"test":0.9806655984652873},{"dataset":"pendigits","label_percent":30,"series":"Tahap 7 · 3.5","name":"head_scarf","picked_by_validation":true,"validation":0.998200745438106,"test":0.9801397736175895},{"dataset":"superconductivity","label_percent":1,"series":"Tahap 7 · 3.5","name":"tabpfn","picked_by_validation":false,"validation":288.0651448567708,"test":318.1292419433594},{"dataset":"superconductivity","label_percent":1,"series":"Tahap 7 · 3.5","name":"head_distill","picked_by_validation":false,"validation":284.5583190917969,"test":314.87498982747394},{"dataset":"superconductivity","label_percent":1,"series":"Tahap 7 · 3.5","name":"head_tabcutmix","picked_by_validation":true,"validation":284.3540954589844,"test":314.48529052734375},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 7 · 3.5","name":"tabpfn","picked_by_validation":false,"validation":150.03161112467447,"test":163.36234029134116},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 7 · 3.5","name":"head_distill","picked_by_validation":false,"validation":155.3275400797526,"test":164.89678955078125},{"dataset":"superconductivity","label_percent":10,"series":"Tahap 7 · 3.5","name":"head_gaussian","picked_by_validation":true,"validation":154.3067626953125,"test":165.39325968424478},{"dataset":"superconductivity","label_percent":30,"series":"Tahap 7 · 3.5","name":"tabpfn","picked_by_validation":false,"validation":105.58319854736328,"test":116.4144058227539},{"dataset":"superconductivity","label_percent":30,"series":"Tahap 7 · 3.5","name":"head_distill","picked_by_validation":false,"validation":105.23486073811848,"test":118.3210678100586},{"dataset":"superconductivity","label_percent":30,"series":"Tahap 7 · 3.5","name":"head_tabcutmix","picked_by_validation":true,"validation":105.41812388102214,"test":118.89878590901692}]}

## 1. Smoke dan timing pada konteks besar

Tiga grup, satu seed, dua view per setting, maksimum 512 baris unlabeled dan 200 langkah.
Konteks Superconductivity 30% tetap memakai 4.501 label sebelum subsetting.
Cek grid, arsip prediksi, kolom QWK terkalibrasi dan cosine angular berpasangan.

In [ ]:
if RUN_SMOKE:
    run("tabpfn_konteks.py", "--out", OUT, "--smoke")
    run("ringkas.py", OUT / "smoke", "--expect", "smoke")

## 2. Grid penuh

Grid utama mencakup 405 run dari 27 grup. Grid kekuatan konteks menambah 63 run
jika `RUN_ABLATIONS = True`. Empat view per setting, maksimal 5.000 baris unlabeled,
3.000 langkah maksimum, dan satu worker per GPU.

Ringkasan memilih konfigurasi di validation per dataset, fraksi label, dan keluarga
eksperimen. Tabel historis memuat RankUp/SSR-RCUS dan hasil TabPFN tahap 7.

In [ ]:
if RUN_FULL:
    selection = "all" if RUN_ABLATIONS else "core"
    run("tabpfn_konteks.py", "--out", OUT, "--names", selection, *FULL_ARGS)
    verify_full = [] if FULL_ARGS else ["--expect", selection]
    run("ringkas.py", OUT / "full", *verify_full)

## 3. Simpan hasil

Arsip memuat JSONL, prediksi validation/test, ringkasan dan diagnostik.
`EXPORT_CACHE` menambahkan embedding cache. Simpan arsip dari tab Output sebelum sesi berakhir.

In [ ]:
export = WORK / "hasil_tabpfn_konteks.zip"
paths = [p for p in OUT.rglob("*") if p.is_file() and p.relative_to(OUT).parts[0] in ("smoke", "full")
         and not p.name.startswith(".") and p.suffix not in (".tmp", ".restore_tmp")
         and (EXPORT_CACHE or "cache" not in p.relative_to(OUT).parts)]
if paths:
    with ZipFile(export, "w", compression=ZIP_DEFLATED, compresslevel=1) as archive:
        for path in sorted(paths):
            archive.write(path, path.relative_to(OUT))
    from IPython.display import FileLink, display
    print(f"{export} ({export.stat().st_size / 2**20:.1f} MiB)")
    display(FileLink(export.name))